# PE6203 Project 3: Verified Results Analysis

This clean notebook reviews the **corrected official-verifier evidence** packaged with the project. It does not call PI, Docker, OpenRouter, or any paid model endpoint.

## Important correction
The original runner used the verifier-wrapper process exit code as its success label. An audit of 23 archived runs showed that this wrapper status was 0 even when internal official verifier tests failed. Therefore, the analysis below defines success only from the internal official verifier outcomes.

## Scope

- Methods: Default, Random Search, Bayesian Optimization, and a custom two-stage cost-aware search.
- Formal validation tasks: `paper-anonymizer`, `edit-pdf`.
- Held-out tasks: `offer-letter-generator`, `court-form-filling`.
- Result: all four methods have verified validation Pass@1 = 0.0; the frozen final configuration has held-out Pass@1 = 0/2.
- Failure evidence: 176 extracted verifier-evidence rows from 23 logs; these are not independent model trials.

In [ ]:
import csv
import json
from pathlib import Path

# Find the package root whether this notebook is opened from its own folder or the package root.
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
ROOT = next(
    p for p in candidates
    if (p / 'results' / 'verified_analysis').exists()
)
ANALYSIS = ROOT / 'results' / 'verified_analysis'

def read_csv(relative_path):
    with (ANALYSIS / relative_path).open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

methods = read_csv('method_comparison_verified.csv')
heldout = read_csv('heldout_test_results_verified.csv')
failure_summary = read_csv('failure_mode_analysis/failure_mode_summary.csv')
audit = json.loads((ANALYSIS / 'official_verifier_audit.json').read_text(encoding='utf-8'))

print('Corrected method results:')
for row in methods:
    print(f"- {row['method']}: verified validation Pass@1={row['verified_validation_pass_at_1']}, cost=${float(row['observed_cumulative_cost_usd']):.6f}")

print('\nCorrected held-out results:')
for row in heldout:
    print(f"- {row['task']}: official verifier Pass@1={row['official_verifier_pass_at_1']}")

print(f"\nAudit: {len(audit)} archived runs; all had old wrapper score 1 but official verifier score 0.")
print('Failure-evidence categories:')
for row in failure_summary:
    print(f"- {row['category']}: {row['failure_evidence_rows']} evidence rows")


## Rebuild the figure

From the package root, run `python scripts/rebuild_verified_analysis.py`. This generates the corrected chart from the stored CSV evidence without a model call or API cost.